In [5]:
# Base packages
import numpy as np
import matplotlib.pyplot as plt
import os
import copy

# I/O
import glob
from skimage import io
from PIL import Image
import h5py
import json
import pickle

# Pytorch
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import ToTensor

# Data Augmentation
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
ia.seed(2)

# Plots
import seaborn as sns
import pandas as pd
from textwrap import wrap
import matplotlib.patheffects as path_effects
import statsmodels.api as sm

# Measurements and Metrics
from sklearn import metrics as skmetrics
from skimage import measure

from scipy.stats import pearsonr

from datetime import datetime

random_state = np.random.RandomState(42)

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO
import cv2
import torchvision.models.segmentation
import torch

In [6]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

In [7]:
def file_list_dict(img_list, gt_list):
    # Returns a dictionary with the list of images and corresponding ground truth
    file_list = {"imgs": img_list,
      "gt": gt_list}
    
    return file_list

In [8]:
class GVHD_Dataset(Dataset):
    def __init__(self, files, mode, transform=None):
        self.transform = transform 
        self.files = files
        self.mode = mode
        
    def __len__(self):
        size=len(self.files["imgs"])
        return size
    
    def __getitem__(self, idx):
        path_img = self.files["imgs"][idx]
        path_mask = self.files["gt"][idx]
        
        if self.mode == 'test':
            image = path_img
            mask = path_mask
        else:
            image = io.imread(path_img)
            mask = io.imread(path_mask)[:,:,0]
        
        return image, mask

In [9]:
def patch_extractor(img, patch_size = 256, stride=32):
    width = img.shape[1]
    height = img.shape[0]
    patch_list = []
    
    if len(img.shape) == 3:
        for j in range(0, width-patch_size+stride, stride):
            for i in range(0, height-patch_size+stride, stride):
                patch_list.extend([img[i:i+patch_size, j:j+patch_size, :]])
    elif len(img.shape) == 2:
        for j in range(0, width-patch_size+stride, stride):
            for i in range(0, height-patch_size+stride, stride):
                patch_list.extend([img[i:i+patch_size, j:j+patch_size]])
    
    return patch_list

In [10]:
def lesion_count(binary_mask):
    mask_labelled = measure.label(binary_mask)
    lesion_number = len(np.unique(mask_labelled))-1
    
    return lesion_number

In [11]:
def patched_inference_list_RCNN(img, model, patch_size = 256, stride=32, include_mask = True):
    pad_val = patch_size-stride
    img_pad = np.pad(img, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant')
    
    patch_list = patch_extractor(img_pad, patch_size, stride)
    
    files_test = file_list_dict(patch_list, patch_list)
    gvhd_data_test = GVHD_Dataset(files_test,'test',composed)
    
    bbox_list = []
    conf_list = []
    mask_list = []
    index_list = []
    edge_list = []
    num_patches_h = len(range(0, img_pad.shape[0]-patch_size+stride, stride))
    num_patches_w = len(range(0, img_pad.shape[1]-patch_size+stride, stride))
    for img_idx in range(len(patch_list)):
        h_index = img_idx % num_patches_h
        w_index = np.floor(img_idx / num_patches_h)
        image, _ = gvhd_data_test[img_idx]
        x_tensor = torch.as_tensor(image, dtype=torch.float32).unsqueeze(0)
        x_tensor = x_tensor.swapaxes(1, 3).swapaxes(2, 3)
        x_tensor = list(image.to(device) for image in x_tensor)
        with torch.no_grad():
            pred = model(x_tensor)
        for i in range(len(pred[0]['masks'])):
            box = pred[0]['boxes'][i].detach().cpu().numpy()
            if (box[0] < 1) or (box[1] < 1) or (box[2] > patch_size-2) or (box[3] > patch_size-2):
                edge_list.append(1)
            else:
                edge_list.append(0)
            box[0] = box[0] + w_index*stride
            box[2] = box[2] + w_index*stride
            box[1] = box[1] + h_index*stride
            box[3] = box[3] + h_index*stride
            bbox_list.append(box)
            conf = pred[0]['scores'][i].detach().cpu().numpy()
            conf = float(conf)
            conf_list.append(conf)
            if include_mask:
                mask = pred[0]['masks'][i,0].detach().cpu().numpy()
                mask_list.append(mask)
                index_list.append((h_index, w_index))
    if include_mask:
        return bbox_list, conf_list, edge_list, mask_list, index_list
    else:
        return bbox_list, conf_list, edge_list

In [12]:
def custom_nms(bounding_boxes, confidence_score, conf_threshold, iou_threshold, edge_list):
    # If no bounding boxes, return empty list
    if len(bounding_boxes) == 0:
        return [], {}
    
    grouping_dict = {}

    # Bounding boxes
    boxes = np.array(bounding_boxes)

    # coordinates of bounding boxes
    start_x = boxes[:, 0]
    start_y = boxes[:, 1]
    end_x = boxes[:, 2]
    end_y = boxes[:, 3]

    # Confidence scores of bounding boxes
    score = np.array(confidence_score)

    # Picked bounding boxes
    picked_boxes = []
    picked_score = []
    indices = []

    # Compute areas of bounding boxes
    areas = (end_x - start_x + 1) * (end_y - start_y + 1)

    # Sort by confidence score of bounding boxes
    order = np.argsort(score)
    order = np.array([i for i in order if confidence_score[i] > conf_threshold])

    # Iterate bounding boxes
    while order.size > 0:
        if edge_list[order[-1]] == 1:
            edge_list[order[-1]] = 0
            np.insert(order, 0, order[-1])
            order = order[0:-1]
            continue
        
        # The index of largest confidence score
        index = order[-1]

        # Pick the bounding box with largest confidence score
        picked_boxes.append(bounding_boxes[index])
        picked_score.append(confidence_score[index])
        indices.append(index)

        # Compute ordinates of intersection-over-union(IOU)
        x1 = np.maximum(start_x[index], start_x[order[:-1]])
        x2 = np.minimum(end_x[index], end_x[order[:-1]])
        y1 = np.maximum(start_y[index], start_y[order[:-1]])
        y2 = np.minimum(end_y[index], end_y[order[:-1]])

        # Compute areas of intersection-over-union
        w = np.maximum(0.0, x2 - x1 + 1)
        h = np.maximum(0.0, y2 - y1 + 1)
        intersection = w * h

        # Compute the ratio between intersection and union
        ratio = intersection / (areas[index] + areas[order[:-1]] - intersection)

        left = np.where(ratio < iou_threshold)
        merge = np.where(ratio >= iou_threshold)
        
        grouping_dict[index] = order[merge]
        order = order[left]

    return indices, grouping_dict

In [13]:
def seg_contour_overlay(mask_list,
                        image,
                        line_size=4,
                        fig_size=(16,16),
                        thresh=0.5,
                        color_list = ['blue','lime','magenta','yellow','red'],
                        save_fig=False,
                        save_name='overlay.jpg',
                        show_fig=True,
                        title='Contour Overlay',
                        save_fig_dpi=150,
                        contour_alpha=0.75
                       ):
    
    if torch.is_tensor(image):
        image = np.moveaxis(image.squeeze().cpu().numpy(), 0, -1)
    
    contour_list = []
    for mask in mask_list:
        contour_list.extend([measure.find_contours(mask, thresh)])
    
    fig, ax = plt.subplots(figsize=fig_size)
    image_new = image.copy()
    image[:,:,0] = image_new[:,:,2]
    image[:,:,2] = image_new[:,:,0]
    ax.imshow(image, cmap=plt.cm.gray, interpolation=None)
    
    color_select = 0
    for mask_contour in contour_list:
        for contour in mask_contour:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=line_size, color=color_list[color_select],alpha=contour_alpha)
        color_select+=1

    ax.set_xticks([])
    ax.set_yticks([])
    plt.title(title, fontsize = 24)
    if save_fig:
        plt.savefig(save_name, dpi=save_fig_dpi, bbox_inches='tight', pad_inches = 0)
    if show_fig:
        plt.show()
    else:
        plt.close()

In [14]:
# w/o visual
def lesion_matching(final_indices, gt_mask, mask_list, index_list, grouping_dict):
    tp = 0
    fp = 0
    fn = 0
    
    patch_size = 256
    stride = 64
    pad_val = patch_size-stride

    padded_gt = np.pad(gt_mask, pad_width=[(pad_val, pad_val),(pad_val, pad_val)], mode='constant')
    gt_mask_labelled = measure.label(padded_gt)
    
    cpy_mask_list = copy.deepcopy(mask_list)
    cpy_index_list = copy.deepcopy(index_list)

    matched_gt_list = []
    for mask_index in final_indices:
        matched_gt = -1
        iou_val = 0
        pr_mask_full = np.zeros(padded_gt.shape)
        count_mask = np.zeros(padded_gt.shape)
        pr_mask_full[int(index_list[mask_index][0]*stride):int(index_list[mask_index][0]*stride+patch_size), int(index_list[mask_index][1]*stride):int(index_list[mask_index][1]*stride+patch_size)] =  mask_list[mask_index]
#         count_mask[int(index_list[mask_index][0]*stride):int(index_list[mask_index][0]*stride+patch_size), int(index_list[mask_index][1]*stride):int(index_list[mask_index][1]*stride+patch_size)] =  1
#         for i in range(len(grouping_dict[mask_index])):
#             pr_mask_full[int(index_list[np.array(grouping_dict[mask_index])[i]][0]*stride):int(index_list[np.array(grouping_dict[mask_index])[i]][0]*stride+patch_size), int(index_list[np.array(grouping_dict[mask_index])[i]][1]*stride):int(index_list[np.array(grouping_dict[mask_index])[i]][1]*stride+patch_size)] =  pr_mask_full[int(index_list[np.array(grouping_dict[mask_index])[i]][0]*stride):int(index_list[np.array(grouping_dict[mask_index])[i]][0]*stride+patch_size), int(index_list[np.array(grouping_dict[mask_index])[i]][1]*stride):int(index_list[np.array(grouping_dict[mask_index])[i]][1]*stride+patch_size)] + mask_list[np.array(grouping_dict[mask_index])[i]]
#             count_mask[int(index_list[grouping_dict[mask_index][i]][0]*stride):int(index_list[grouping_dict[mask_index][i]][0]*stride+patch_size), int(index_list[grouping_dict[mask_index][i]][1]*stride):int(index_list[grouping_dict[mask_index][i]][1]*stride+patch_size)] += (mask_list[grouping_dict[mask_index][i]] > 0)

        count_mask = np.maximum(count_mask, np.ones(padded_gt.shape))
        pr_mask_full = pr_mask_full / count_mask
        pr_mask_full = pr_mask_full > 0.5
        
        for region_label in range (1,len(np.unique(gt_mask_labelled))):
            if region_label not in matched_gt_list:
                current_region_mask = np.zeros(padded_gt.shape)
                current_region_mask[gt_mask_labelled == region_label] = 1
                curr_iou = np.sum(pr_mask_full[current_region_mask == 1])/(np.sum(pr_mask_full) + np.sum(gt_mask_labelled == region_label) - np.sum(pr_mask_full[current_region_mask == 1]))
                if curr_iou > 0:
                    if curr_iou > iou_val:
                        iou_val = curr_iou
                        matched_gt = region_label
        if matched_gt != -1:
            tp += 1
            matched_gt_list.append(matched_gt)
        else:
            fp += 1
    
    fn = len(np.unique(gt_mask_labelled)) - len(matched_gt_list)

    return tp, fp, fn

In [15]:
unet_prediction_dir = 'final_UNetPlusPlus_output' + os.sep
unet_prediction_list = sorted(glob.glob(unet_prediction_dir + os.sep + '**' + os.sep + '*.h5', recursive=True))

In [23]:
# w visual - TODO
def lesion_matching_mutual(final_indices, gt_mask, mask_list, index_list, grouping_dict, current_photo_name, current_img, current_img_path):
    current_filename = current_img_path.replace('training_images', 'final_UNetPlusPlus_output')
    current_filename = current_filename.replace('.png', '_PredictionResults.h5')
    print(current_filename)
    print(current_img_path)

    with h5py.File(current_filename, 'r') as hdf:
        current_gt = np.asarray(hdf['gt_mask'])
        current_predict_mask = np.asarray(hdf['pr_mask_optimum'])
    print()
    
    shared_tp = 0
    shared_fp = 0
    shared_fn = 0
    rcnn_only_fp = 0
    rcnn_only_fn = 0
    unet_only_fp = 0
    unet_only_fn = 0
    
    patch_size = 256
    stride = 64
    pad_val = patch_size-stride

    padded_unet = np.pad(current_predict_mask, pad_width=[(pad_val, pad_val),(pad_val, pad_val)], mode='constant')
    unet_labelled = measure.label(padded_unet)
    padded_gt = np.pad(gt_mask, pad_width=[(pad_val, pad_val),(pad_val, pad_val)], mode='constant')
    gt_mask_labelled = measure.label(padded_gt)
    
    cpy_mask_list = copy.deepcopy(mask_list)
    cpy_index_list = copy.deepcopy(index_list)
    
    tp_map = np.zeros(padded_gt.shape)
    shared_error_map = np.zeros(padded_gt.shape)
    unet_error_map = np.zeros(padded_gt.shape)
    rcnn_error_map = np.zeros(padded_gt.shape)

    matched_gt_list = []
    matched_unet_list = []
    for mask_index in final_indices:
        matched_gt = -1
        iou_val = 0
        pr_mask_full = np.zeros(padded_gt.shape)
        count_mask = np.zeros(padded_gt.shape)
        pr_mask_full[int(index_list[mask_index][0]*stride):int(index_list[mask_index][0]*stride+patch_size), int(index_list[mask_index][1]*stride):int(index_list[mask_index][1]*stride+patch_size)] =  mask_list[mask_index]
        pr_mask_full = pr_mask_full > 0.5
        
        for region_label in range (1,len(np.unique(gt_mask_labelled))):
            if region_label not in matched_gt_list:
                current_region_mask = np.zeros(padded_gt.shape)
                current_region_mask[gt_mask_labelled == region_label] = 1
                curr_iou = np.sum(pr_mask_full[current_region_mask == 1])/(np.sum(pr_mask_full) + np.sum(gt_mask_labelled == region_label) - np.sum(pr_mask_full[current_region_mask == 1]))
                if curr_iou > 0.01:
                    if curr_iou > iou_val:
                        iou_val = curr_iou
                        matched_gt = region_label
        if matched_gt != -1:
            iou_val = 0
            matched_unet = -1
            for pred_label in range (1,len(np.unique(unet_labelled))):
                unet_mask_full = np.zeros(padded_gt.shape)
                unet_mask_full[unet_labelled == pred_label] = 1
                current_region_mask = np.zeros(padded_gt.shape)
                current_region_mask[gt_mask_labelled == matched_gt] = 1
                curr_iou = np.sum(unet_mask_full[current_region_mask == 1])/(np.sum(unet_mask_full) + np.sum(gt_mask_labelled == region_label) - np.sum(unet_mask_full[current_region_mask == 1]))
                if curr_iou > 0.01:
                    if curr_iou > iou_val:
                        iou_val = curr_iou
                        matched_unet = pred_label
            image_region = np.zeros(padded_gt.shape)
            image_region[gt_mask_labelled == matched_gt] = 1
            if matched_unet != -1:
                shared_tp += 1
                tp_map = np.maximum(image_region, tp_map)
                matched_gt_list.append(matched_gt)
                matched_unet_list.append(matched_unet)
            else:
                unet_only_fn += 1
                matched_gt_list.append(matched_gt)
                unet_error_map = np.maximum(image_region, unet_error_map)
        else:
            iou_val = 0
            matched_unet = -1
            for pred_label in range (1,len(np.unique(unet_labelled))):
                unet_mask_full = np.zeros(padded_gt.shape)
                unet_mask_full[unet_labelled == pred_label] = 1
                curr_iou = np.sum(unet_mask_full[pr_mask_full == 1])/(np.sum(unet_mask_full) + np.sum(np.sum(pr_mask_full)) - np.sum(unet_mask_full[pr_mask_full == 1]))
                if curr_iou > 0.01:
                    if curr_iou > iou_val:
                        iou_val = curr_iou
                        matched_unet = pred_label
            if matched_unet != -1:
                shared_fp += 1
                image_region = np.zeros(padded_gt.shape)
                image_region[unet_labelled == matched_unet] = 1
                shared_error_map = np.maximum(image_region, shared_error_map)
                matched_unet_list.append(matched_unet)
            else:
                rcnn_only_fp += 1
                rcnn_error_map = np.maximum(pr_mask_full, rcnn_error_map)
    
    for i in range (1,len(np.unique(gt_mask_labelled))):
        if i not in matched_gt_list:
            current_region_mask = np.zeros(padded_gt.shape)
            current_region_mask[gt_mask_labelled == i] = 1
            iou_val = 0
            matched_unet = -1
            for j in range (1,len(np.unique(unet_labelled))):
                if j not in matched_unet_list:
                    unet_mask_full = np.zeros(padded_gt.shape)
                    unet_mask_full[unet_labelled == j] = 1
                    curr_iou = np.sum(unet_mask_full[current_region_mask == 1])/(np.sum(unet_mask_full) + np.sum(current_region_mask) - np.sum(unet_mask_full[current_region_mask == 1]))
                    if curr_iou > 0.01:
                        if curr_iou > iou_val:
                            iou_val = curr_iou
                            matched_unet = pred_label
            image_region = np.zeros(padded_gt.shape)
            image_region[gt_mask_labelled == i] = 1
            if matched_unet != -1:
                rcnn_only_fn += 1
                rcnn_error_map = np.maximum(image_region, rcnn_error_map)
                matched_unet_list.append(matched_unet)
            else:
                shared_fn += 1
                shared_error_map = np.maximum(image_region, shared_error_map)
                
    unet_only_fp = len(np.unique(unet_labelled)) - len(matched_unet_list)
    for i in range (1,len(np.unique(unet_labelled))):
        if i not in matched_unet_list:
            current_region_mask = np.zeros(padded_gt.shape)
            current_region_mask[gt_mask_labelled == i] = 1
            unet_error_map = np.maximum(current_region_mask, unet_error_map)
    
    tp_map = tp_map[(256-64):(64-256),(256-64):(64-256)]
    shared_error_map = shared_error_map[(256-64):(64-256),(256-64):(64-256)]
    unet_error_map = unet_error_map[(256-64):(64-256),(256-64):(64-256)]
    rcnn_error_map = rcnn_error_map[(256-64):(64-256),(256-64):(64-256)]
    
    seg_contour_overlay([tp_map,shared_error_map,unet_error_map,rcnn_error_map],
        current_img,
        line_size=3,
        fig_size=(16,16),
        thresh=0.5,
        color_list = ['cyan','yellow','red','pink'],
        save_fig=True,
        save_name='Mutual_Error_Map' + os.sep + current_photo_name + '.jpg',
        title='shared tp: ' + str(shared_tp) + '\nshared error: ' + str(shared_fp+shared_fn) + '\nrcnn error: ' + str(rcnn_only_fp+rcnn_only_fn) + '\nunet error: ' + str(unet_only_fp+unet_only_fn),
        show_fig=False,
        save_fig_dpi=150,
        contour_alpha=1)
    
    print(shared_tp, shared_fp, shared_fn, rcnn_only_fp, rcnn_only_fn, unet_only_fp, unet_only_fn)

    return shared_tp, shared_fp, shared_fn, rcnn_only_fp, rcnn_only_fn, unet_only_fp, unet_only_fn

In [24]:
def MPX_nms_thresholded(img_masked, gt_mask, bbox_list, conf_list, mask_list, index_list, edge_list, image_name):
    conf_threshold_list = np.arange(0.6, 1, 0.05)
#     iou_threshold_list = [0.05]
    iou_threshold_list = np.arange(0.05, 0.21, 0.05)
    
    patch_size = 256
    stride = 64
    pad_val = patch_size-stride
    
    bg_masked_indicies = []
    bg_masked_img  = np.pad(img_masked, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant')
    for les_index in range(len(conf_list)):
        if np.sum(bg_masked_img[int(bbox_list[les_index][1]):int(bbox_list[les_index][3]), int(bbox_list[les_index][0]):int(bbox_list[les_index][2])]) > 0:
            bg_masked_indicies.append(les_index)

    counter=0
    pr_count_list = np.zeros((len(iou_threshold_list), len(conf_threshold_list)))
    pr_metric_list = np.zeros((len(iou_threshold_list), len(conf_threshold_list)))
    current_gt_lesions = lesion_count(gt_mask)

    for (i, iou_threshold) in enumerate(iou_threshold_list):
        for (j, conf_threshold) in enumerate(conf_threshold_list):
            indices, grouping_dict = custom_nms(bbox_list, conf_list, conf_threshold, iou_threshold, edge_list)
            final_indices = np.intersect1d(indices, bg_masked_indicies)

            tp, fp, fn = lesion_matching(final_indices, gt_mask, mask_list, index_list, grouping_dict)

            pr_count_list[i][j] = len(final_indices)
            pr_metric_list[i][j] = tp/(tp+fp) + tp/(tp+fn)

            if counter % 10 == 0:
                print('Threshold number ' + str(counter) + r' / ' + str(len(conf_threshold_list)*len(iou_threshold_list)))
            counter+=1

    return pr_count_list, current_gt_lesions, pr_metric_list, iou_threshold_list, conf_threshold_list

In [25]:
torch.cuda.empty_cache()
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

composed = {
    'image':transforms.Compose([ToTensor()]),
    'mask':transforms.Compose([ToTensor()]),
}

In [26]:
# Data folder (laptop)
data_root = ''

optimum_thresh_dir = r'.\final_RCNN_thresh' + os.sep
save_root = r'.\final_RCNN_output_40' + os.sep
os.makedirs(optimum_thresh_dir, exist_ok=True)
os.makedirs(save_root, exist_ok=True)

model_directory = '.' + os.sep + 'saved_models_patch_RCNN' + os.sep
model_list = sorted(glob.glob(model_directory + '*/39.torch'))

full_img_dir = data_root + 'training_images' + os.sep
full_gt_dir = data_root + 'ground_truth' + os.sep

# optimum_thresh_dir = r'.\220604_MPX_JAMADerm_OptimumThresh' + os.sep

### NOTE: FAKED MASKED DIRECTORY FOR NOW - CALCULATIONS NOT ACUTALLY USED YET
full_masked_img_dir = full_gt_dir #data_root + 'images_bgmasked' + os.sep + 'ZC' + os.sep

print(len(model_list), model_list[0])

18 .\saved_models_patch_RCNN\051301\39.torch


In [27]:
dir_list = sorted(glob.glob(full_img_dir + os.sep + '*'))

img_paths_dict = {}
gt_paths_dict = {}
masked_img_paths_dict = {}
patient_name_list = []
for dir_no in range(len(dir_list)):
    directory = dir_list[dir_no]
    patient_name = os.path.basename(directory)
    patient_name_list.extend([patient_name])
    
    img_paths_dict[patient_name] = sorted(glob.glob(directory + os.sep + '*.png'))
    gt_paths_dict[patient_name] = sorted(glob.glob(full_gt_dir + patient_name_list[dir_no] + os.sep+ '*.png'))
    masked_img_paths_dict[patient_name] = sorted(glob.glob(full_masked_img_dir + patient_name_list[dir_no] + os.sep+ '*.png'))

In [28]:
optimum_iou_thresh_jsons = sorted(glob.glob(optimum_thresh_dir + 'OptimumThreshiou_*.json'))
optimum_conf_thresh_jsons = sorted(glob.glob(optimum_thresh_dir + 'OptimumThreshconf_*.json'))

optimum_iou_thresh_dict = {}
optimum_conf_thresh_dict = {}
for json_file in optimum_iou_thresh_jsons:
    path_split = os.path.basename(json_file).split('patient-')
    current_patient_name = path_split[1][:-5]
    
    with open(json_file, 'r') as f:
        optimum_iou_thresh_dict[current_patient_name] = json.load(f)

for json_file in optimum_conf_thresh_jsons:
    path_split = os.path.basename(json_file).split('patient-')
    current_patient_name = path_split[1][:-5]
    
    with open(json_file, 'r') as f:
        optimum_conf_thresh_dict[current_patient_name] = json.load(f)

print(optimum_iou_thresh_dict)
print(optimum_conf_thresh_dict)

{'051301': 0.05, '051302': 0.05, '051307': 0.05, '051308': 0.05, '051309': 0.05, '051310': 0.05, '213': 0.05, '221': 0.05, '227': 0.05, '236': 0.05, '238': 0.05, '239': 0.05, '247': 0.05, '248': 0.05, 'Kole1': 0.05, 'Kole2': 0.05, 'Kole3': 0.05, 'MPX subject coming from Dekese (Not enrolled)': 0.05}
{'051301': 0.7940196078431374, '051302': 0.81421568627451, '051307': 0.8140522875816997, '051308': 0.8381862745098042, '051309': 0.807761437908497, '051310': 0.8150326797385623, '213': 0.8329411764705883, '221': 0.8084150326797388, '227': 0.8173202614379088, '236': 0.825637254901961, '238': 0.8092320261437911, '239': 0.829199346405229, '247': 0.8158496732026146, '248': 0.8071732026143793, 'Kole1': 0.8103758169934643, 'Kole2': 0.8290849673202617, 'Kole3': 0.8389379084967322, 'MPX subject coming from Dekese (Not enrolled)': 0.8157516339869283}


In [29]:
contour_save_root = r'.\Error_RCNN_contour' + os.sep
os.makedirs(contour_save_root, exist_ok=True)

# list_of_test_patients = list(optimum_iou_thresh_dict.keys())
list_of_test_patients = patient_name_list

RCNN_metric_arr = []
for patient_name in list_of_test_patients:
    model_found = False
    for current_model_name in model_list:
        current_model_name_split = current_model_name.split('\\')
        if current_model_name_split[-2] == patient_name:
            pretrained_model_name = current_model_name
            model_found = True
    
    if model_found:
        print('Loading pretrained model "' + pretrained_model_name + '"')
        device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')   # train on the GPU or on the CPU, if a GPU is not available
        pretrained_model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)  # load an instance segmentation model pre-trained pre-trained on COCO
        in_features = pretrained_model.roi_heads.box_predictor.cls_score.in_features  # get number of input features for the classifier
        pretrained_model.roi_heads.box_predictor = FastRCNNPredictor(in_features,num_classes=2)  # replace the pre-trained head with a new one
        pretrained_model.load_state_dict(torch.load(pretrained_model_name))
        pretrained_model.to(device)
        pretrained_model.eval()
    else:
        print('No model found for patient ' + patient_name)
        break
    
    # Predict on held-out patient
    for img_no in range(len(img_paths_dict[patient_name])): #predict on all photos for current patient
        current_img_path = img_paths_dict[patient_name][img_no]
        current_gt_path = gt_paths_dict[patient_name][img_no]
        
        current_image_name = os.path.basename(current_img_path)[:-4]
        
        print(' Testing image "' + current_image_name + '"')
        
        test_img = np.asarray(cv2.imread(current_img_path))
        test_gt = np.asarray(cv2.imread(current_gt_path))[:,:,0:3]
        test_gt_binary = np.max(test_gt,2).astype('bool')
        
        bbox_list, conf_list, edge_list, mask_list, index_list = patched_inference_list_RCNN(test_img, pretrained_model, patch_size=256, stride=64)
        indicies, grouping_dict = custom_nms(bbox_list, conf_list, optimum_conf_thresh_dict[patient_name], optimum_iou_thresh_dict[patient_name], edge_list)

        patch_size = 256
        stride = 64
        pad_val = patch_size-stride
        
        bg_masked_indicies = []
        bg_masked_name = current_img_path.replace('training_images', 'images_bgmasked')
        bg_masked_name = bg_masked_name.replace('.png', '_masked.png')
        bg_masked_img = cv2.imread(bg_masked_name)
        bg_masked_img = np.pad(bg_masked_img, pad_width=[(pad_val, pad_val),(pad_val, pad_val),(0, 0)], mode='constant')
        for les_index in indicies:
            if np.sum(bg_masked_img[int(bbox_list[les_index][1]):int(bbox_list[les_index][3]), int(bbox_list[les_index][0]):int(bbox_list[les_index][2])]) > 0:
                bg_masked_indicies.append(les_index)
        
        lesion_count = len(bg_masked_indicies)
        shared_tp, shared_fp, shared_fn, rcnn_only_fp, rcnn_only_fn, unet_only_fp, unet_only_fn = lesion_matching_mutual(bg_masked_indicies, test_gt_binary, mask_list, index_list, grouping_dict, current_image_name, test_img, current_img_path)
        new_element = {}
        new_element['shared_tp'] = shared_tp
        new_element['shared_fp'] = shared_fp
        new_element['shared_fn'] = shared_fn
        new_element['rcnn_only_fp'] = rcnn_only_fp
        new_element['rcnn_only_fn'] = rcnn_only_fn
        new_element['unet_only_fp'] = unet_only_fp
        new_element['unet_only_fn'] = unet_only_fn
        RCNN_metric_arr.append(new_element)

Loading pretrained model ".\saved_models_patch_RCNN\051301\39.torch"


C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


 Testing image "051301 D2_06"
final_UNetPlusPlus_output\051301\051301 D2_06_PredictionResults.h5
training_images\051301\051301 D2_06.png

18 2 7 5 4 1 4
 Testing image "051301 D2_07"
final_UNetPlusPlus_output\051301\051301 D2_07_PredictionResults.h5
training_images\051301\051301 D2_07.png

45 1 15 7 8 4 6
 Testing image "051301 D2_08"
final_UNetPlusPlus_output\051301\051301 D2_08_PredictionResults.h5
training_images\051301\051301 D2_08.png

22 3 32 3 5 5 1
 Testing image "051301 D8_02"
final_UNetPlusPlus_output\051301\051301 D8_02_PredictionResults.h5
training_images\051301\051301 D8_02.png

121 1 20 9 3 0 7
Loading pretrained model ".\saved_models_patch_RCNN\051302\39.torch"


C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


 Testing image "051302 D2_01"
final_UNetPlusPlus_output\051302\051302 D2_01_PredictionResults.h5
training_images\051302\051302 D2_01.png

15 12 13 7 5 8 2
 Testing image "051302 D2_02"
final_UNetPlusPlus_output\051302\051302 D2_02_PredictionResults.h5
training_images\051302\051302 D2_02.png

13 5 9 7 3 3 2
Loading pretrained model ".\saved_models_patch_RCNN\051307\39.torch"


...

 Testing image "Kole2 01"
final_UNetPlusPlus_output\Kole2\Kole2 01_PredictionResults.h5
training_images\Kole2\Kole2 01.png

10 2 9 13 1 2 4
 Testing image "Kole2 02"
final_UNetPlusPlus_output\Kole2\Kole2 02_PredictionResults.h5
training_images\Kole2\Kole2 02.png

10 1 18 12 7 3 2
 Testing image "Kole2 03"
final_UNetPlusPlus_output\Kole2\Kole2 03_PredictionResults.h5
training_images\Kole2\Kole2 03.png

8 2 12 4 1 5 1
Loading pretrained model ".\saved_models_patch_RCNN\Kole3\39.torch"


C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


 Testing image "Kole3 01"
final_UNetPlusPlus_output\Kole3\Kole3 01_PredictionResults.h5
training_images\Kole3\Kole3 01.png

10 1 20 9 0 1 17
 Testing image "Kole3 02"
final_UNetPlusPlus_output\Kole3\Kole3 02_PredictionResults.h5
training_images\Kole3\Kole3 02.png

4 0 22 7 4 4 15
Loading pretrained model ".\saved_models_patch_RCNN\MPX subject coming from Dekese (Not enrolled)\39.torch"


C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\ProgramData\Anaconda3\envs\mip3.8\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


 Testing image "MPX subject coming from Dekese (Not enrolled) 1"
final_UNetPlusPlus_output\MPX subject coming from Dekese (Not enrolled)\MPX subject coming from Dekese (Not enrolled) 1_PredictionResults.h5
training_images\MPX subject coming from Dekese (Not enrolled)\MPX subject coming from Dekese (Not enrolled) 1.png

162 25 37 12 29 16 9
 Testing image "MPX subject coming from Dekese (Not enrolled) 2"
final_UNetPlusPlus_output\MPX subject coming from Dekese (Not enrolled)\MPX subject coming from Dekese (Not enrolled) 2_PredictionResults.h5
training_images\MPX subject coming from Dekese (Not enrolled)\MPX subject coming from Dekese (Not enrolled) 2.png

197 20 18 18 9 11 35
